# MaxCut on PIAST-Q — a QLauncher example

This example solves **MaxCut** with a **linear-ramp QAOA** on the **AQT** trapped-ion backend
(PIAST-Q) using QLauncher, and shows the workflow across three execution modes:

1. the **noiseless** AQT simulator,
2. the AQT **calibrated-noise** simulator,
3. the **real device** (ARNICA).

It follows QLauncher's `Problem → Algorithm → Backend → Launcher` pattern:

| | |
|---|---|
| **Problem** | `MaxCut(graph)` |
| **Algorithm** | `LinearRampQAOA(...)` (in `src/lr_qaoa.py`) |
| **Backend** | `AQTBackend('local_simulator' \| 'backendv1v2' \| 'device')` |
| **Launcher** | `QLauncher(problem, algorithm, backend).run()` |

`LinearRampQAOA` is a small, deterministic QAOA: instead of training the circuit on the device, it
normalises the cost Hamiltonian and optimises the two linear-ramp parameters *classically* on the
exact statevector, then samples the backend **once**. On top of the QLauncher `Result`, the helpers
in `src/` add scoring against the exact optimum, random-cut and Goemans–Williamson baselines, 95%
bootstrap confidence intervals, an all-to-all vs limited-connectivity comparison, and an
integration-cost report. A Slurm + QCG-PilotJob batch runner is in `piastq_hpc/`.

**Requirements:** Python ≥ 3.11, `qiskit < 2`, `pip install qlauncher`.

## 1. Problem — MaxCut on a graph

In [ ]:
import networkx as nx

from qlauncher import QLauncher
from qlauncher.problems import MaxCut
from qlauncher.routines.qiskit import AQTBackend

from src.lr_qaoa import LinearRampQAOA
from src.hybrid_analysis import score_result, plot_modes, exact_max_cut, connectivity_comparison
from src.integration_cost import integration_cost

graph = nx.gnp_random_graph(10, 0.5, seed=7)
problem = MaxCut(graph, instance_name='piastq_n10')
problem.visualize()

## 2. Algorithm — LinearRampQAOA

In [ ]:
algorithm = LinearRampQAOA(p_grid=(1, 2, 3), shots=2000)

## 3. Mode 1 — AQT simulator (noiseless)

`auto_transpile_level=0` translates circuits to AQT-native gates without optimisation.

In [ ]:
backend_ideal = AQTBackend('local_simulator', auto_transpile_level=0)
result_ideal = QLauncher(problem, algorithm, backend_ideal).run()
print(result_ideal)

## 4. Mode 2 — AQT simulator with AQT's noise model

The noisy offline simulator from `qiskit-aqt-provider` is passed via the `backendv1v2` mode.

In [ ]:
from qiskit_aqt_provider import AQTProvider
from qiskit_aqt_provider.aqt_resource import OfflineSimulatorResource

_prov = AQTProvider('OFFLINE')
_rid = _prov.get_backend('offline_simulator_no_noise').resource_id
noisy_sim = OfflineSimulatorResource(_prov, _rid, with_noise_model=True)

backend_noise = AQTBackend('backendv1v2', backendv1v2=noisy_sim, auto_transpile_level=0)
result_noise = QLauncher(problem, algorithm, backend_noise).run()
print(result_noise)

## 5. Score against the exact optimum

`score_result` reports the approximation ratio, `P(ratio ≥ 0.9)` (the fraction of shots within
90% of the optimum), and `P(optimum)`, each with a 95% bootstrap CI, plus the random-cut and
Goemans–Williamson baselines.

In [ ]:
opt, opt_bitstring = exact_max_cut(graph)
print('exact max cut =', opt)

scores = {
    'AQT ideal': score_result(result_ideal, graph, optimum=opt),
    'AQT noise': score_result(result_noise, graph, optimum=opt),
}
for name, s in scores.items():
    print(f"{name}: mean={s['mean_ratio']:.3f}  P(>=0.9)={s['p_good']:.2f} CI{s['p_good_ci95']}  P(opt)={s['p_opt']:.3f}")

plot_modes(scores, opt, 'modes_comparison.png')

Example output for this instance (seed 7):

![modes comparison](modes_comparison.png)

## 6. Integration cost

Classical / quantum wall-time split, quantum-duty fraction, and the number of device submissions
(one) versus a naive iteratively-trained QAOA.

In [ ]:
print('AQT ideal:', integration_cost(result_ideal))
print('AQT noise:', integration_cost(result_noise))

## 7. All-to-all vs limited connectivity (same backend)

The same QAOA layer is run natively (all-to-all) and forced onto a line (SWAP-routed) via
`backend.sample_circuit`. The depth auto-reduces so the routed circuit stays under the device's
2000-operations-per-circuit limit (`p_used`).

In [ ]:
conn = connectivity_comparison(backend_noise, graph, p=3, shots=1500)
print('depth used:', conn['p_used'],
      '| two-qubit gates  native:', conn['native_2q_gates'], ' routed(line):', conn['routed_2q_gates'])
print('P(>=0.9)  native all-to-all:', round(conn['native_all_to_all']['p_good'], 3),
      ' forced onto grid:', round(conn['forced_onto_grid']['p_good'], 3))

## 8. Mode 3 — the real PIAST-Q (ARNICA)

Same code, `AQTBackend('device', ...)`; put the ARNICA token in a `.env` file (`AQT_TOKEN=...`).

In [ ]:
# backend_hw = AQTBackend('device', dotenv_path='./.env', auto_transpile_level=0)
# result_hw = QLauncher(problem, algorithm, backend_hw).run()
# scores['PIAST-Q'] = score_result(result_hw, graph, optimum=opt)
# plot_modes(scores, opt, 'modes_comparison.png')

## Batch sweep on HPC (Slurm + QCG-PilotJob)

`piastq_hpc/run.py` runs one configuration and writes `results/<tag>.json`; `submit_sweep.sh`
launches `sweep_qcgpj.py`, which uses **QCG-PilotJob** to distribute the runs across a Slurm
allocation; `plot_sweep.py` turns the results into a figure.

```bash
cd piastq_hpc
sbatch submit_sweep.sh          # aqt_ideal + aqt_noise over several seeds
python plot_sweep.py            # -> sweep_comparison.png
```

Example output from a 5-seed sweep on the cluster (one QPU submission per run):

![sweep comparison](piastq_hpc/sweep_comparison.png)